# НЕ УСПЕЛ

In [2]:
import pandas as pd
from gensim.models import Word2Vec

import pandas as pd
import torch
from train_utils import ItemDataset, QueriesDataset, get_device, evaluate
from models import TwoTower
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

seed = 42

d:\projects\avito_retr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Загрузка данных

In [ ]:
queries_df = pd.read_parquet('preprocessed/train_queries.parquet')
items_df = pd.read_parquet('preprocessed/train_items_hybrid.parquet')

# торопился, но столько обработать будет долго, поэтому
items_df['item_description_raw_norm'] = items_df["item_description_raw_norm"].fillna("").str[:300]

### Загрузка трансформера

In [4]:
device = get_device()
transformer = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device=device,
    cache_folder="models/hf"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8289.39it/s]


### Загрузка W2V

In [5]:
w2v = Word2Vec.load('models/w2v/word2vec_128.model')

#### если еще не обучали (получится одинаково)

In [5]:
item_corpus = items_df['item_title_raw_norm']
query_corpus = queries_df['search_query_norm']

sentences = pd.concat([
    query_corpus,
    item_corpus
]).tolist()
sentences = [w.tolist() for w in sentences]

In [ ]:
vec_size = 128
w2v = Word2Vec(
    sentences=sentences,
    vector_size=vec_size,
    window=5,
    min_count=2,
    sg=1,
    negative=10,
    workers=8,
    epochs=10
)

w2v.save(f'models/w2v/word2vec_{vec_size}.model')

### Train

In [22]:
g = torch.Generator()
g.manual_seed(seed)

item_dataset = ItemDataset(w2v.wv, items_df=items_df, encoder_big=transformer)

train_df, val_df = train_test_split(queries_df, test_size=0.1, random_state=seed)
train_dataset = QueriesDataset(queries_df=train_df, item_dataset=item_dataset)
val_dataset = QueriesDataset(queries_df=val_df, item_dataset=item_dataset, mode='train')

Use Word2Vec


item_title_raw_norm: 100%|██████████| 344825/344825 [00:04<00:00, 73100.15it/s]


Use SentenceTransformer


Batches:   9%|▉         | 474/5388 [02:19<24:05,  3.40it/s]


KeyboardInterrupt: 

In [ ]:
train_dataloader = DataLoader(train_dataset, 
                              num_workers=0, 
                              shuffle=True, 
                              batch_size=1024,
                              generator=g)

In [ ]:
model = TwoTowerHybrid()

In [ ]:
device = get_device()
model.to(device)
optimizer = torch.optim.Adam(model.parameters())

temperature = 0.1
n_epochs = 10
batches_per_epoch = 300
best_val_recall = -torch.inf
max_bad_epochs = 3

for epoch in range(n_epochs):
    model.train()   
    batch_iter = iter(train_dataloader)
    pbar = tqdm(range(batches_per_epoch))
    loss_sum = 0
    for batch_idx in pbar:
        q_emb, i_emb, context, item_desc, item_ids, _, _ = [tensor.to(device) for tensor in next(batch_iter)]
        
        q, i = model((q_emb, i_emb, context, item_desc))
        logits = (q @ i.T) / temperature
        
        same_item = item_ids[:, None] == item_ids[None, :]
        diag = torch.arange(logits.size(0), device=device)
        same_item[diag, diag] = False

        logits[same_item] = -torch.inf
        targets = torch.arange(logits.size(0), device=device)
        loss = F.cross_entropy(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        pbar.set_description(f"Epoch {epoch} --- loss: {loss_sum / (batch_idx + 1)} | ")

    recall = evaluate(queries_dataset=val_dataset, items_dataset=item_dataset, model=model, device=device, k=50, filters=False)
    print(recall)
    if best_val_recall > recall['Recall@50']:
        max_bad_epochs -= 1
    else:
        best_val_recall = recall['Recall@50']
        torch.save(model.state_dict(), "models/w2v/two_tower_hybrid_128.pth")
    if max_bad_epochs == 0:
        break